# nb_06_idempotent_silver (stretch): a Silver rental build that is safe to run again
**Task 6.** Running the build twice must not duplicate rows or bring a quarantined rental back, and a new day's file must add only its own rentals.

**How it works:** every run reads the files it is given, keeps the latest row per `rental_id`, applies the same null-safe quality rule, then **MERGEs on `rental_id`** into the target tables. A merge on the business key updates in place instead of appending, so a re-run or a re-sent file cannot inflate the counts.

This notebook writes to separate `_idem` tables so it never disturbs the tables from Task 2.

In [ ]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

BASE = "Files/raw"            # same value as in the other notebooks
T_OK, T_BAD = "silver_rentals_idem", "silver_rentals_quarantine_idem"

ALL_FILES = [f"{BASE}/rentals/rentals_2026-06-{d:02d}.csv" for d in range(1, 31)] + \
            [f"{BASE}/rentals/rentals_2026-06-10_resend.csv"]

## The build, written once

In [ ]:
def read_raw(paths):
    return (spark.read.option("header", True).option("inferSchema", False).csv(paths)
            .withColumn("ingested_at", F.current_timestamp())
            .withColumn("source_file", F.col("_metadata.file_path")))

def transform(raw):
    """Same rules as nb_02_silver. Returns (valid, quarantine)."""
    raw = (raw.withColumn("rental_id", F.trim("rental_id"))
              .withColumn("customer_id", F.trim("customer_id"))
              .withColumn("depot_code", F.upper(F.trim("depot_code")))
              .withColumn("asset_id", F.upper(F.trim("asset_id")))
              .withColumn("_rid", F.monotonically_increasing_id()))
    w = (Window.partitionBy("rental_id")
               .orderBy(F.col("ingested_at").desc(), F.col("source_file").desc(), F.col("_rid").desc()))
    r = raw.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "_rid")

    rt = F.lower(F.trim("rental_type"))
    r = (r.withColumn("rental_type", F.when(rt == "standard", "standard").when(rt == "priority", "priority"))
          .withColumn("checkout_ts", F.to_timestamp(F.trim("checkout_ts")))
          .withColumn("checkin_ts", F.to_timestamp(F.when(F.trim("checkin_ts") != "", F.trim("checkin_ts")))))
    bad = F.coalesce(F.col("checkin_ts") < F.col("checkout_ts"), F.lit(False))     # null-safe
    return r.filter(~bad), r.filter(bad)

def upsert(df, table):
    if not spark.catalog.tableExists(table):
        df.write.format("delta").saveAsTable(table)
        return
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), "t.rental_id = s.rental_id")      # the business key, NOT a surrogate or a timestamp
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

def run_build(paths):
    ok, bad = transform(read_raw(paths))
    upsert(ok, T_OK)
    upsert(bad, T_BAD)
    # a rental that is quarantined can never sit in the clean table
    spark.sql(f"DELETE FROM {T_OK} WHERE rental_id IN (SELECT rental_id FROM {T_BAD})")

def counts():
    return spark.table(T_OK).count(), spark.table(T_BAD).count()

## Experiment: start from scratch with every file except 30 June

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T_OK}")
spark.sql(f"DROP TABLE IF EXISTS {T_BAD}")

before_30 = [p for p in ALL_FILES if "2026-06-30" not in p]
run_build(before_30)
c1 = counts()
print("after first run (30 June not loaded yet) : silver_rentals, quarantine =", c1)

## Proof 1: run the same build again, the counts must be identical

In [ ]:
run_build(before_30)
c2 = counts()
print("after running the same build twice       :", c2)
assert c1 == c2, "NOT idempotent: counts changed on a re-run"
print("OK  identical counts on re-run")

## Proof 2: add one new day (30 June), only that day's valid rentals appear

In [ ]:
day_ok, day_bad = transform(read_raw([f"{BASE}/rentals/rentals_2026-06-30.csv"]))
expected_new = day_ok.count()
print("valid rentals in the 30 June file:", expected_new, "| quarantined in that file:", day_bad.count())

run_build(ALL_FILES)
c3 = counts()
print("after adding 30 June                      :", c3)
assert c3[0] - c2[0] == expected_new, (c3[0] - c2[0], expected_new)
assert c3[1] - c2[1] == day_bad.count()
print(f"OK  silver_rentals grew by exactly {expected_new}, the valid rentals of that day")

## Proof 3: a full re-run matches Task 2, and the quarantine never comes back into Silver

In [ ]:
run_build(ALL_FILES)
c4 = counts()
print("after one more full run                   :", c4)
assert c3 == c4
assert c4 == (942, 3), c4          # same as nb_02_silver
overlap = spark.sql(f"SELECT COUNT(*) FROM {T_OK} o JOIN {T_BAD} q ON o.rental_id = q.rental_id").first()[0]
assert overlap == 0
print("OK  matches the Task 2 counts (942 / 3) and no quarantined rental is in the clean table")

**Write-up (paste into your README):** the build merges on `rental_id`, the natural key. A matched row is updated in place and an unmatched row is inserted, so running it twice, or loading the re-sent 10 June file, cannot create a second copy of a rental. The dedupe-by-latest step before the merge guarantees one source row per key, and the final `DELETE ... IN (quarantine)` guarantees a quarantined rental never reappears in the clean table. A merge on the wrong key, such as `source_file` or a timestamp, would be worse than an append because it would silently overwrite or duplicate good rows.